# Sweeping a parameter

One scenario per value of `approved.limit`, over a million synthetic records.
`batch_size` sets how many records run at a time: nothing holds the whole
input's results, and the `Progress` callback reports after every batch.

In [1]:
import polars as pl
from IPython.display import display
from decider.experiments import (Bars, Changes, Describe, Divergence, Experiment, Progress, RunStatus,
                                 ValueCounts)
from example_projects.experiments.credit_pipeline import build


def show(spec):
    """Draw a plot's Vega-Lite spec inline; JupyterLab and VS Code render it interactively."""
    display({"application/vnd.vegalite.v5+json": spec}, raw=True)


ENTRY = "example_projects.experiments.credit_pipeline:build"
pipeline = build()
df = pl.DataFrame({
    "income": [1000.0, 2000.0, 1500.0, 800.0],
    "debt": [250.0, 500.0, 700.0, 250.0],
})
df

shape: (4, 2)
┌────────┬───────┐
│ income ┆ debt  │
│ ---    ┆ ---   │
│ f64    ┆ f64   │
╞════════╪═══════╡
│ 1000.0 ┆ 250.0 │
│ 2000.0 ┆ 500.0 │
│ 1500.0 ┆ 700.0 │
│ 800.0  ┆ 250.0 │
└────────┴───────┘


In [2]:
import numpy as np

rng = np.random.default_rng(0)
loans = pl.DataFrame({"income": rng.uniform(500, 5000, 1_000_000), "debt": rng.uniform(0, 2000, 1_000_000)})

exp = Experiment("limit-sweep", pipeline, loans, entry=ENTRY, batch_size=250_000)
for limit in (0.25, 0.3, 0.35, 0.4, 0.45):
    exp.scenario(f"limit_{limit}", params={"credit": {"approved": {"limit": limit}}})
exp.metric(Divergence(column="approved"), ValueCounts(column="approved"))
exp.plot(Changes(metric="divergence"), Bars(metric="value_counts"))
exp.callback(Progress())

result = exp.run()
result.status

limit-sweep: 250,000/1,000,000 records, 0.2s
limit-sweep: 500,000/1,000,000 records, 0.3s
limit-sweep: 750,000/1,000,000 records, 0.3s
limit-sweep: 1,000,000/1,000,000 records, 0.3s
<RunStatus.COMPLETED: 'completed'>


`limit_0.35` is the pipeline's own default, so it changes nothing against the
baseline; every other value moves approvals one way or the other.

In [3]:
{name: s.changed for name, s in result.summaries["divergence"].items()}

{'baseline': 0, 'limit_0.25': 137350, 'limit_0.3': 68753, 'limit_0.35': 0, 'limit_0.4': 68429, 'limit_0.45': 132410}


In [4]:
show(result.plots["changes"])
show(result.plots["bars"])

<VegaLite 5 chart>

<VegaLite 5 chart>